# Lesson 4.2 · Regularization, scaling and multiclass

scikit-learn's `LogisticRegression` is **regularized by default**: what it hands over is not the minimum of the log-loss but the minimum of the
log-loss plus a penalty on the size of the coefficients. This changes three things that the module 1 theory had already given you for
linear regression and that carry over here.

**What you will learn**

- The role of `C` (careful: it is the **inverse** of the regularization) and the difference between L2 and L1.
- Why scaling changes the model.
- How to go from 2 to K classes (softmax, one-vs-rest) and which metrics to use.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target                                   # 30 nodule measurements, y = 1 benign
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(X.shape, "| benign:", round(y.mean(), 3))

## `C` and the penalty

The minimized function is $C \cdot \text{log-loss} + \text{penalty}(\mathbf{w})$, with penalty $\lVert\mathbf{w}\rVert_2^2/2$ (L2, default)
or $\lVert\mathbf{w}\rVert_1$ (L1). **Small `C` = strong regularization.** L2 shrinks the coefficients; L1 zeroes some of them
(feature selection), but requires a solver that supports it (`liblinear` or `saga`).

In [ ]:
rows = []
for c in [0.001, 0.01, 0.1, 1, 10, 100]:
    for pen in ["l2", "l1"]:
        m = make_pipeline(StandardScaler(), LogisticRegression(C=c, penalty=pen, solver="liblinear", max_iter=2000)).fit(X_tr, y_tr)
        rows.append({"C": c, "penalty": pen, "nonzero coefs": int((m[-1].coef_ != 0).sum()),
                     "coef norm": np.abs(m[-1].coef_).sum().round(2), "test AUC": roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]).round(4)})
pd.DataFrame(rows).pivot(index="C", columns="penalty")

Read the table: with large `C` the coefficients grow and the AUC is almost unchanged; with `C = 0.001` L1 zeroes **all** the coefficients (AUC 0.5, that is chance)
and L2 shrinks them a lot but the model still works. With intermediate `C` L1 keeps only 7-13 features out of 30 (with `C = 0.1` and `C = 1`) with an AUC almost equal to that of L2 which uses them all; L2 never zeroes anything. `C` must be chosen in cross-validation, like any
hyperparameter (`LogisticRegressionCV` does it for you).

## Why scaling changes the model

The penalty sums the coefficients of **all** the features with the same weight. If a feature is measured in large units (meters instead of
millimeters) its coefficient needs to be small to produce the same effect, and so it is penalized less: changing the unit of
measure changes the solution. Without regularization scaling would not change the predictions, but it would make the solver converge more slowly.

In [ ]:
X_mm = X_tr.copy(); X_mm["mean area"] = X_mm["mean area"] / 1000            # same information, another unit (area / 1000)
X_mm_te = X_te.copy(); X_mm_te["mean area"] = X_mm_te["mean area"] / 1000

for name, (a, b) in {"original unit": (X_tr, X_te), "area / 1000": (X_mm, X_mm_te)}.items():
    without = LogisticRegression(C=0.1, max_iter=5000).fit(a, y_tr)
    with_scaling = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(a, y_tr)
    print(f"{name:18s} | without scaling: AUC {roc_auc_score(y_te, without.predict_proba(b)[:, 1]):.4f} | with scaling: AUC {roc_auc_score(y_te, with_scaling.predict_proba(b)[:, 1]):.4f}")

With scaling the two cases coincide to the fourth decimal (after standardization the features are the same, whatever the unit of measure). Without scaling the result changes with the units: here only a little, because the dataset is easy, but the penalty hits features with a small scale harder.
The rule of thumb: **always standardize the features before a regularized logistic** (inside the `Pipeline`, never before the split).

## More than two classes

With K classes the logistic becomes **multinomial** (softmax): a vector of coefficients for every class and probabilities that sum to 1
(`multi_class="multinomial"`, the default with `lbfgs`). Alternatively `OneVsRestClassifier` trains K independent binary models.
The coefficient is therefore a `(K, p)` matrix.

In [ ]:
digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits.data, digits.target, test_size=0.3, stratify=digits.target, random_state=0)
multi = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000)).fit(Xd_tr, yd_tr)
print("shape of coef_:", multi[-1].coef_.shape, "| accuracy:", round(multi.score(Xd_te, yd_te), 4))
print("the probabilities of a digit sum to", multi.predict_proba(Xd_te[:1]).sum().round(6))

from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(multi, Xd_te, yd_te, cmap="Blues", colorbar=False); plt.show()

        With more classes the confusion matrix is 10×10: the errors concentrate in a few pairs (digits that look alike). To summarize in
        a single number you need averages: **macro** (mean of the per-class metrics, every class weighs the same), **micro** (global counts,
        it coincides with the accuracy) or **weighted**. You choose by looking at what you care about: the rare classes (macro) or the total (micro).

        > **Common mistake.** Comparing metrics of models with the L1 and L2 penalty without redoing the choice of `C`: the best `C` for one penalty is not the best for the other.

        ## Check your understanding

        <details><summary>Why is `C=1e10` a way to get the logistic "without regularization"?</summary>

Because `C` multiplies the log-loss: with a huge `C` the penalty weighs almost nothing. It is the portable way across scikit-learn versions (`penalty=None` or `"none"` change name and are removed depending on the version).

</details>

<details><summary>Do I also need to scale the one-hot dummies before a regularized logistic?</summary>

It is a choice. Scaling them brings them to the same scale as the numerics (fair penalty), but makes them less interpretable. Leaving them at 0/1 is common and works well; the important thing is to scale the numerics with very different scales.

</details>